In [1]:
# Use the regressor trained in B3_2 to make full prediction dataframe
# convert modeled dataframe to one GeoTIFF per month

# V1 - 2026-07-29 - create code - jpk

# Workflow
# 1. Load config, model, and spatial template
# 2. Iterate over days in the period range
# 3. Per day: load rows from parquet, predict, reconstruct DataArray
# 4. Align DataArray to template extent via reproject_match
# 5. Save as float32 GeoTIFF


In [2]:
#libraries
import os
import glob
import joblib
import json
import numpy as np
import pandas as pd
import xarray as xr
import rioxarray as rxr
from tqdm import tqdm

start_year, start_month, start_day = 2001, 1, 1
end_year,   end_month, end_day   = 2005, 12, 31

#paths
out_base = fr"..\Data_Temp\Processed\Daily_bias_correction"
model_path = os.path.join(out_base, r"Saved_regressors")
template = rxr.open_rasterio(r"..\Data_Temp\Datasets\CHIRTS\CHIRTS_tmax_20010101.tif")

tmax_temp_dir = os.path.join(out_base, 'Temp_tmax')
tmax_yearly_parquet_dir = os.path.join(tmax_temp_dir, 'Tmax_yearly_parquet')
tmax_final_pred_dir = os.path.join(out_base, "Final_products", "Tmax")

tmin_temp_dir = os.path.join(out_base, 'Temp_tmin')
tmin_yearly_parquet_dir = os.path.join(tmin_temp_dir, 'Tmin_yearly_parquet')
tmin_final_pred_dir = os.path.join(out_base, "Final_products", "Tmin")

for d in [tmax_temp_dir, tmax_yearly_parquet_dir, tmax_final_pred_dir,
    tmin_temp_dir, tmin_yearly_parquet_dir, tmin_final_pred_dir]:
    if not os.path.exists(d):
        os.makedirs(d)
        
# read predictor features and model for each variable 
def get_model_features(t_var:str):
    metadata_file = glob.glob(os.path.join(model_path, f"{t_var}_*_feature_metadata.json"))[0]
    with open(metadata_file, 'r') as f:
            metadata = json.load(f)
    predictor_features = metadata['input_features']
    model_type = metadata['model_type']
    return model_type, predictor_features

model_type_tmax, predictor_features_tmax = get_model_features("tmax")
model_type_tmin, predictor_features_tmin = get_model_features("tmin")

In [3]:
print(pd.read_parquet(r"C:\Users\A S U S\Documents\Study\ClimateData\CD-Workspace\Data_Temp\Processed\Daily_bias_correction\Temp_tmax\Tmax_yearly_parquet\full_2001.parquet").columns)

Index(['x', 'y', 'spatial_ref', 'chirts_tmax', 'har_tmax', 'era5l_tmax',
       'gldas_air_temp', 'gldas_lwdown', 'gldas_specific_humidity',
       'gldas_surface_pressure', 'gldas_swdown', 'gldas_wind_speed', 'ik_tmax',
       'elev_mean', 'elev_std', 'elev_max', 'elev_min', 'slope_mean',
       'aspect_mean', 'elev_smoothed', 'fetch_1000m_dist', 'fetch_1000m_dir',
       'fetch_500m_dist', 'fetch_500m_dir', 'local_relief', 'doy_sin',
       'doy_cos', 'year', 'month', 'day', 'season_num'],
      dtype='object')


In [4]:
#  Main prediction loop 

# For each day:
#   0. Load the model
#   1. Skip if the output raster already exists (safe to re-run)
#   2. Read only that day's rows from parquet (memory-efficient)
#   3. Predict with the model 
#   4. Pivot predictions back to a 2-D DataArray indexed by (y, x)
#   5. Write CRS / nodata / transform, then align to full template extent
#   6. Save as float32 GeoTIFF

day_range = pd.period_range(
    start = f"{start_year}-{start_month:02d}-{start_day:02d}",
    end   = f"{end_year}-{end_month:02d}-{end_day:02d}",
    freq  = "D",
)

for t_variable in ['tmax', 'tmin']:
    print(f"\n Predicting daily rasters for {t_variable}==========================================")
    if t_variable == 'tmax':
        _, predictor_features = model_type_tmax, predictor_features_tmax
        model = joblib.load(os.path.join(model_path, f'{t_variable}_{model_type_tmax}.pkl'))
    else:
        _, predictor_features = model_type_tmin, predictor_features_tmin
        model = joblib.load(os.path.join(model_path, f'{t_variable}_{model_type_tmin}.pkl'))

    for p in tqdm(day_range, desc="Predict rasters"):
        year, month, day = p.year, p.month, p.day
        if t_variable == "tmax":
            outpath = os.path.join(tmax_final_pred_dir, f"tmax_{year}{month:02d}{day:02d}.tif")
            df_year_path = os.path.join(tmax_yearly_parquet_dir, f"full_{year}.parquet")
        else:
            outpath = os.path.join(tmin_final_pred_dir, f"tmin_{year}{month:02d}{day:02d}.tif")
            df_year_path = os.path.join(tmin_yearly_parquet_dir, f"full_{year}.parquet")

        # Skip if already done 
        if os.path.exists(outpath):
            tqdm.write(f"  [{year}-{month:02d}{day:02d}] already exists, skipping.")
            continue

        # Load this day's rows from parquet 
        df_day = pd.read_parquet(
            df_year_path,
            filters=[("year", "==", year), ("month", "==", month), ("day", "==", day)],
        )

        if df_day.empty:
            tqdm.write(f"  [{year}-{month:02d}{day:02d}] no rows found, skipping.")
            continue

        # Predict
        df_day[f"{t_variable}_predicted"] = model.predict(
            df_day[predictor_features]
        )

        # Make a rio xarray raster
        da = df_day.set_index(["y", "x"])[f"{t_variable}_predicted"].to_xarray()
        da = (
            da
            .rio.write_crs(template.rio.crs)
            .rio.write_nodata(-9999)
            .rio.write_transform(template.rio.transform())
        )
        da = da.rio.reproject_match(template)
        da.rio.to_raster(outpath, dtype="float32")

    print(f"\nDone. All rasters saved for {t_variable}")


 Predicting daily rasters for tmax==========================================


Predict rasters: 100%|██████████| 1826/1826 [03:19<00:00,  9.16it/s]



Done. All rasters saved for tmax

 Predicting daily rasters for tmin==========================================


Predict rasters: 100%|██████████| 1826/1826 [04:06<00:00,  7.39it/s]


Done. All rasters saved for tmin
